In [1]:
"""
Phase-space reconstruction and Lyapunov-time diagnostics -- NO4.

Reproduces the pipeline that produced DK1's tau=1, m=8, h~114h (Section
4.5 / Appendix G.1-G.2), run on NO4's own price-difference series, so
that m is estimated for NO4 rather than inherited from DK1 or DK2.

Three stages:
  1. Average Mutual Information (AMI) over a range of lags -> select tau
     as the first local minimum (paper's stated criterion).
  2. Cao's E1(d)/E2(d) method over a range of embedding dimensions d ->
     E1 stabilizing (ratio E1(d+1)/E1(d) -> 1) indicates the minimum
     sufficient embedding dimension; E2(d) ~ 1 across d indicates
     predominantly stochastic (not low-dimensional chaotic) dynamics,
     consistent with the paper's own reading of E2 for DK1/SE4/NO3-NO4.
  3. Wolf-type nearest-neighbour orbit-divergence procedure -> estimate
     the maximal Lyapunov exponent lambda_max at each of several m
     values (not just the one E1 "settles" on), so the resulting
     Lyapunov-time estimate can be checked for the same kind of
     m-sensitivity documented for DK2 (Table 9: h swung from ~126h at
     m=8 to ~338h at m=14).

NOTE ON SCOPE: this uses NO4's own OPSD column, not the NO3/NO4 average
used in the Experiment 4 alpha-sensitivity script. Keep this consistent
with whichever series your forecasting tables end up using for this area.

Does not depend on any prior notebook/kernel state. Run top to bottom in
a fresh process, with time_series_60min_singleindex.csv in the working
directory. Requires scikit-learn (for nearest-neighbour search).
"""

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# ============================================================
# CONFIG
# ============================================================

OPSD_PATH = "time_series_60min_singleindex.csv"
OPSD_DATE_COL = "utc_timestamp"
TARGET_COLUMN = "NO_4_price_day_ahead"
AREA_LABEL = "NO4"

AMI_MAX_LAG = 30          # search lags 1..30 for the first AMI local minimum
AMI_BINS = 32             # histogram bins for the mutual-information estimate

CAO_D_MAX = 29             # sweep d = 1..29, matching the DK2 sensitivity check's range
CAO_THEILER = 5            # exclude this many temporally adjacent points as "self" neighbours

# m values to run the full Wolf-type Lyapunov estimate at (expensive step,
# so run on a handful of candidates rather than every d in the Cao sweep).
# Includes 8 (DK1's value, for direct comparison) plus a spread above it,
# mirroring the DK2 sensitivity table's grid.
LYAPUNOV_M_GRID = [8, 14, 20, 26]

WOLF_EVOLVE_STEPS = 10      # steps to evolve each pair before checking/replacing
WOLF_MIN_SEPARATION = 1e-6  # minimum allowed initial separation (avoid div-by-zero)
WOLF_MAX_SEPARATION_FACTOR = 10.0  # replace neighbour once separation grows this much

SAMPLING_INTERVAL_HOURS = 1.0  # NO4 is hourly data -> each embedding step = 1 hour


# ============================================================
# Data loading
# ============================================================
def load_price_diffs():
    data = pd.read_csv(OPSD_PATH, usecols=[OPSD_DATE_COL, TARGET_COLUMN])
    data.columns = ["date", "close"]
    data["close"] = pd.to_numeric(data["close"], errors="coerce")
    data = data.dropna(subset=["close"]).reset_index(drop=True)
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data = data.dropna(subset=["date"])
    data = data.sort_values("date").drop_duplicates(subset="date", keep="last").reset_index(drop=True)

    prices = data["close"].astype(float).values
    r = np.diff(prices)
    print(f"{AREA_LABEL}: {len(r)} usable price-difference points "
          f"({data['date'].min()} to {data['date'].max()})")
    return r


# ============================================================
# Stage 1: Average Mutual Information -> select tau
# ============================================================
def average_mutual_information(x, lag, bins=AMI_BINS):
    x1 = x[:-lag]
    x2 = x[lag:]
    c_xy, xedges, yedges = np.histogram2d(x1, x2, bins=bins)
    p_xy = c_xy / c_xy.sum()
    p_x = p_xy.sum(axis=1, keepdims=True)
    p_y = p_xy.sum(axis=0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = p_xy / (p_x * p_y)
        term = p_xy * np.log(ratio)
    term[~np.isfinite(term)] = 0.0
    return term.sum()


def select_tau(x, max_lag=AMI_MAX_LAG):
    amis = [average_mutual_information(x, lag) for lag in range(1, max_lag + 1)]
    amis = np.array(amis)
    print("\n--- AMI by lag ---")
    for lag, val in zip(range(1, max_lag + 1), amis):
        print(f"  lag={lag:2d}  AMI={val:.5f}")
    # first local minimum: first lag where AMI stops decreasing
    tau = 1
    for i in range(1, len(amis)):
        if amis[i] > amis[i - 1]:
            tau = i  # lags are 1-indexed; amis[i-1] corresponds to lag=i
            break
    else:
        tau = max_lag
        print("  No local minimum found within max_lag -- defaulting to max_lag. "
              "Consider increasing AMI_MAX_LAG.")
    print(f"Selected tau (first AMI local minimum) = {tau}")
    return tau, amis


# ============================================================
# Stage 2: Cao's E1(d) / E2(d)
# ============================================================
def _embed(x, d, tau):
    n = len(x) - (d - 1) * tau
    if n < 2:
        raise ValueError(f"Series too short to embed at d={d}, tau={tau}.")
    return np.array([x[i:i + d * tau:tau] for i in range(n)])


def cao_criterion(x, tau, d_max=CAO_D_MAX, theiler=CAO_THEILER):
    """Returns E1(d) and E2(d) for d = 1..d_max, following Cao (1997)."""
    e_vals, estar_vals = [], []
    for d in range(1, d_max + 1):
        Yd = _embed(x, d, tau)
        Yd1 = _embed(x, d + 1, tau)
        n = min(len(Yd), len(Yd1))
        Yd, Yd1 = Yd[:n], Yd1[:n]

        nn = NearestNeighbors(n_neighbors=theiler + 2).fit(Yd)
        dists, idxs = nn.kneighbors(Yd)

        a_vals, adist_vals = [], []
        for i in range(n):
            # pick nearest neighbour excluding self and temporally close points
            nbr = None
            for k in range(1, dists.shape[1]):
                cand = idxs[i, k]
                if abs(cand - i) > theiler:
                    nbr = cand
                    dist_d = dists[i, k]
                    break
            if nbr is None or nbr >= n:
                continue
            dist_d1 = np.max(np.abs(Yd1[i] - Yd1[nbr]))
            if dist_d <= 1e-12:
                continue
            a_vals.append(dist_d1 / dist_d)
            adist_vals.append(abs(x[i + d * tau] - x[nbr + d * tau]) if i + d * tau < len(x) and nbr + d * tau < len(x) else np.nan)

        e_vals.append(np.nanmean(a_vals) if a_vals else np.nan)
        estar_vals.append(np.nanmean(adist_vals) if adist_vals else np.nan)

    E = np.array(e_vals)
    Estar = np.array(estar_vals)
    E1 = np.full(d_max, np.nan)
    E2 = np.full(d_max, np.nan)
    E1[:-1] = E[1:] / E[:-1]
    E2[:-1] = Estar[1:] / Estar[:-1]
    return E1, E2


def run_cao(x, tau, d_max=CAO_D_MAX):
    E1, E2 = cao_criterion(x, tau, d_max=d_max)
    print(f"\n--- Cao's E1(d)/E2(d), tau={tau} ---")
    print(f"{'d':>3} {'E1(d)':>10} {'E1 ratio':>10} {'E2(d)':>10}")
    prev = None
    for d in range(1, d_max):
        ratio = E1[d - 1] / prev if prev not in (None, 0) and not np.isnan(prev) else np.nan
        print(f"{d:3d} {E1[d-1]:10.4f} {ratio:10.4f} {E2[d-1]:10.4f}")
        prev = E1[d - 1]
    print("E1 -> 1 as d increases indicates saturation (sufficient embedding dimension).")
    print("E2 ~ 1 across d indicates predominantly stochastic dynamics (not low-dim chaos).")
    return E1, E2


# ============================================================
# Stage 3: Wolf-type maximal Lyapunov exponent
# ============================================================
def wolf_lyapunov(x, m, tau, evolve_steps=WOLF_EVOLVE_STEPS,
                   min_sep=WOLF_MIN_SEPARATION, max_sep_factor=WOLF_MAX_SEPARATION_FACTOR,
                   theiler=CAO_THEILER, dt_hours=SAMPLING_INTERVAL_HOURS):
    """Standard Wolf et al. (1985) nearest-neighbour divergence procedure.
    Returns lambda_max in nats per sample (matching Table 8's units)."""
    Y = _embed(x, m, tau)
    n = len(Y)
    if n < 100:
        raise ValueError(f"Series too short for m={m}, tau={tau} (only {n} embedded points).")

    nn_finder = NearestNeighbors(n_neighbors=min(50, n - 1)).fit(Y)

    def nearest_neighbour(i, exclude_idx=None):
        dists, idxs = nn_finder.kneighbors(Y[i:i + 1], n_neighbors=min(50, n - 1))
        for dist, idx in zip(dists[0], idxs[0]):
            if idx == i or abs(idx - i) <= theiler:
                continue
            if exclude_idx is not None and idx == exclude_idx:
                continue
            if dist >= min_sep:
                return idx, dist
        return None, None

    log_growth_sum = 0.0
    total_evolve_time = 0
    i = 0
    j, L0 = nearest_neighbour(i)
    if j is None:
        raise RuntimeError("Could not find an initial neighbour pair -- series may be too short/degenerate.")

    max_start = n - evolve_steps - 1
    while i < max_start:
        i_next = i + evolve_steps
        j_next = j + evolve_steps
        if j_next >= n:
            j, L0 = nearest_neighbour(i)
            if j is None or j + evolve_steps >= n:
                break
            continue

        L1 = np.max(np.abs(Y[i_next] - Y[j_next]))
        if L1 <= 1e-12 or L0 <= 1e-12:
            i = i_next
            j, L0 = nearest_neighbour(i)
            if j is None:
                break
            continue

        log_growth_sum += np.log(L1 / L0)
        total_evolve_time += evolve_steps

        # replace neighbour if separation has grown too much
        if L1 > max_sep_factor * L0 or j_next >= n - 1:
            j_new, L0_new = nearest_neighbour(i_next, exclude_idx=j_next)
            if j_new is None:
                i = i_next
                j, L0 = nearest_neighbour(i)
                if j is None:
                    break
                continue
            j, L0 = j_new, L0_new
        else:
            j, L0 = j_next, L1

        i = i_next

    if total_evolve_time == 0:
        raise RuntimeError("No valid evolution steps accumulated -- check evolve_steps/min_sep settings.")

    lambda_max = log_growth_sum / (total_evolve_time * dt_hours)
    return lambda_max


def run_lyapunov_sweep(x, tau, m_grid=LYAPUNOV_M_GRID):
    print(f"\n--- Wolf-type maximal Lyapunov exponent, tau={tau} ---")
    print(f"{'m':>3} {'lambda_max (nat/sample)':>26} {'Lyapunov time h (hours)':>26}")
    rows = []
    for m in m_grid:
        try:
            lam = wolf_lyapunov(x, m=m, tau=tau)
            h = 1.0 / lam if lam > 0 else np.inf
            print(f"{m:3d} {lam:26.6f} {h:26.2f}")
            rows.append({"m": m, "lambda_max": lam, "lyapunov_time_hours": h})
        except Exception as e:
            print(f"{m:3d}  FAILED: {e}")
            rows.append({"m": m, "lambda_max": np.nan, "lyapunov_time_hours": np.nan})
    return pd.DataFrame(rows)


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    r = load_price_diffs()

    tau, amis = select_tau(r)

    E1, E2 = run_cao(r, tau)

    results_df = run_lyapunov_sweep(r, tau)
    results_df.to_csv(f"lyapunov_m_sensitivity_{AREA_LABEL.lower()}.csv", index=False)
    print(f"\nSaved: lyapunov_m_sensitivity_{AREA_LABEL.lower()}.csv")

    print("\n===== SUMMARY =====")
    print(f"Selected tau = {tau}")
    print("Check where E1(d) stabilizes above (ratio E1(d+1)/E1(d) close to 1) to read off")
    print("the minimum sufficient embedding dimension m, the same way DK1's m=8 and")
    print("DK2's m=14 (Table 9) were read off -- do not just take LYAPUNOV_M_GRID's")
    print("first entry (8) on faith without checking this against the E1 curve for NO4.")
    print(results_df.to_string(index=False))
    print("\nIf the Lyapunov-time estimate swings substantially across the m grid (as it")
    print("did for DK2, ~126h at m=8 to ~338h at m=14), report the sensitivity explicitly")
    print("rather than picking a single m, per the open item already flagged for DK1/DK2.")

NO4: 50385 usable price-difference points (2015-01-01 01:00:00+00:00 to 2020-09-30 22:00:00+00:00)

--- AMI by lag ---
  lag= 1  AMI=0.06765
  lag= 2  AMI=0.02046
  lag= 3  AMI=0.00660
  lag= 4  AMI=0.00601
  lag= 5  AMI=0.00658
  lag= 6  AMI=0.00500
  lag= 7  AMI=0.00317
  lag= 8  AMI=0.00356
  lag= 9  AMI=0.00425
  lag=10  AMI=0.00630
  lag=11  AMI=0.00854
  lag=12  AMI=0.00952
  lag=13  AMI=0.00797
  lag=14  AMI=0.00549
  lag=15  AMI=0.00384
  lag=16  AMI=0.00385
  lag=17  AMI=0.00288
  lag=18  AMI=0.00387
  lag=19  AMI=0.00422
  lag=20  AMI=0.00383
  lag=21  AMI=0.00442
  lag=22  AMI=0.01409
  lag=23  AMI=0.03944
  lag=24  AMI=0.07273
  lag=25  AMI=0.04172
  lag=26  AMI=0.01486
  lag=27  AMI=0.00470
  lag=28  AMI=0.00417
  lag=29  AMI=0.00470
  lag=30  AMI=0.00409
Selected tau (first AMI local minimum) = 4

--- Cao's E1(d)/E2(d), tau=4 ---
  d      E1(d)   E1 ratio      E2(d)
  1     0.3344        nan     0.3003
  2     0.3025     0.9046     0.7017
  3     0.4238     1.4011     0.8